# LeetCode 1171 — Remove Zero Sum Consecutive Nodes from Linked List bằng Python

Link đề: https://leetcode.com/problems/remove-zero-sum-consecutive-nodes-from-linked-list/ · Độ khó: **Medium** · Chủ đề: Hash Table, Linked List

Skeleton LeetCode:

```python
# Definition for singly-linked list.
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
class Solution:
    def removeZeroSumSublists(self, head: ListNode | None) -> ListNode | None:
        ...
```

Ghi chú: đáp án là **đầu (head) của danh sách liên kết** sau khi lặp lại việc xoá các **đoạn nút liên tiếp có tổng bằng 0** cho đến khi không còn đoạn nào. **Có thể có nhiều đáp án đúng** (ví dụ `[1,2,-3,3,1]` → `[3,1]` hoặc `[1,2,1]`), nên notebook kiểm tra bằng **hàm kiểm tra hợp lệ** chứ không so khớp một đáp án cố định. Điểm cốt lõi: **tổng tiền tố (prefix sum)** — hai vị trí có cùng tổng tiền tố thì đoạn giữa chúng có tổng 0.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: xoá lặp lại các đoạn liên tiếp tổng 0; kết quả cuối không còn đoạn liên tiếp nào tổng 0.
2. Nắm công cụ: tự dựng `ListNode`, chuyển list ↔ linked list, **nút giả (dummy node)**, tổng tiền tố, `dict` ánh xạ tổng tiền tố → nút.
3. Đi từ **mô phỏng xoá lặp lại** → **mỗi nút tìm đoạn tổng 0 bắt đầu từ nó** → **hash map hai lượt** → **hash map một lượt + dọn dẹp (stack)**.
4. Hiểu invariant: "cùng tổng tiền tố ⇒ đoạn giữa tổng 0", và vì sao trỏ tới **lần xuất hiện cuối** của mỗi tổng tiền tố xoá được mọi đoạn trong một lượt.
5. Biết cách test bài **nhiều đáp án**: kiểm tra (a) không còn đoạn tổng 0, (b) kết quả đạt được bằng một chuỗi phép xoá hợp lệ.

## 2. Đề bài & ví dụ

Cho `head` của một danh sách liên kết đơn. Ta **lặp lại** việc xoá các dãy nút **liên tiếp** có tổng bằng `0` cho đến khi không còn dãy nào như vậy. Trả về `head` của danh sách cuối cùng. **Chấp nhận mọi đáp án hợp lệ.**

**Ví dụ 1:** `[1,2,-3,3,1]` → `[3,1]` (cũng chấp nhận `[1,2,1]`: xoá `[-3,3]` thay vì `[1,2,-3]`).

**Ví dụ 2:** `[1,2,3,-3,4]` → `[1,2,4]`.

**Ví dụ 3:** `[1,2,3,-3,-2]` → `[1]` (xoá `[3,-3]`, rồi `[2,-2]`; hoặc một lần xoá `[2,3,-3,-2]`).

**Ràng buộc:** `1..1000` nút; `-1000 <= node.val <= 1000`.

In [1]:
import random
import time
from collections import deque
from typing import List, Optional


class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


def build(vals):
    dummy = ListNode()
    cur = dummy
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
    return dummy.next


def to_list(head):
    out = []
    while head:
        out.append(head.val)
        head = head.next
    return out


examples = [
    ([1, 2, -3, 3, 1], [3, 1]),
    ([1, 2, 3, -3, 4], [1, 2, 4]),
    ([1, 2, 3, -3, -2], [1]),
]
for vals, want in examples:
    print(vals, "->", want, "| build/to_list:", to_list(build(vals)))

[1, 2, -3, 3, 1] -> [3, 1] | build/to_list: [1, 2, -3, 3, 1]
[1, 2, 3, -3, 4] -> [1, 2, 4] | build/to_list: [1, 2, 3, -3, 4]
[1, 2, 3, -3, -2] -> [1] | build/to_list: [1, 2, 3, -3, -2]


## 3. Kiến thức nền

### 3.1 Danh sách liên kết đơn và nút giả (dummy)

Mỗi `ListNode` có `val` và `next`. Xoá một đoạn `[a..b]` = cho nút **trước** `a` trỏ thẳng tới nút **sau** `b`. Rắc rối: nếu đoạn bắt đầu từ `head`, không có "nút trước". **Dummy node** đặt trước `head` giải quyết: luôn có nút trước, và đáp án là `dummy.next`.

In [2]:
def delete_range(head, i, j):
    """Xoá các nút ở vị trí i..j (0-based) bằng dummy."""
    dummy = ListNode(0, head)
    before = dummy
    for _ in range(i):
        before = before.next
    after = before
    for _ in range(j - i + 2):
        after = after.next
    before.next = after
    return dummy.next

print(to_list(delete_range(build([1, 2, -3, 3, 1]), 0, 2)))   # xoá cả head
print(to_list(delete_range(build([1, 2, -3, 3, 1]), 2, 3)))
assert to_list(delete_range(build([1, 2, -3, 3, 1]), 0, 2)) == [3, 1]

[3, 1]
[1, 2, 1]


### 3.2 Tổng tiền tố (prefix sum)

`P[k] = a[0] + … + a[k-1]`, với `P[0] = 0`. Tổng đoạn `a[i..j] = P[j+1] − P[i]`. Vậy **đoạn tổng 0 ⟺ `P[i] == P[j+1]`**: hai vị trí có cùng tổng tiền tố.

In [3]:
a = [1, 2, -3, 3, 1]
P = [0]
for x in a:
    P.append(P[-1] + x)
print("P =", P)
pairs = [(i, j) for i in range(len(P)) for j in range(i + 1, len(P)) if P[i] == P[j]]
print("cặp (i, j) cùng tổng tiền tố -> đoạn a[i:j] tổng 0:", [(i, j, a[i:j]) for i, j in pairs])

P = [0, 1, 3, 0, 3, 4]
cặp (i, j) cùng tổng tiền tố -> đoạn a[i:j] tổng 0: [(0, 3, [1, 2, -3]), (2, 4, [-3, 3])]


### 3.3 `dict` ánh xạ tổng tiền tố → nút

Khi duyệt danh sách, `seen[prefix] = node` ghi nhớ nút có tổng tiền tố đó. Gán lại cùng khoá sẽ **ghi đè** → sau một lượt duyệt, `seen[p]` là nút **cuối cùng** có tổng `p`.

In [4]:
seen = {}
for i, p in enumerate([0, 1, 3, 0, 3, 4]):
    seen[p] = i
print(seen)    # 0 -> 3, 3 -> 4: lần xuất hiện cuối
assert seen[0] == 3 and seen[3] == 4

{0: 3, 1: 1, 3: 4, 4: 5}


### 3.4 Kiểm tra "không còn đoạn tổng 0"

Một list không còn đoạn liên tiếp tổng 0 ⟺ mọi tổng tiền tố (kể cả `P[0] = 0`) **đôi một khác nhau**. Đây là cách kiểm tra `O(n)`.

In [5]:
def has_zero_segment(vals):
    s, seen = 0, {0}
    for v in vals:
        s += v
        if s in seen:
            return True
        seen.add(s)
    return False

print(has_zero_segment([3, 1]), has_zero_segment([1, 2, -3, 3, 1]), has_zero_segment([0]))
assert not has_zero_segment([3, 1]) and has_zero_segment([0])

False True True


### 3.5 Kiểm tra hợp lệ cho bài nhiều đáp án

Với input nhỏ, ta liệt kê **mọi** trạng thái cuối có thể đạt được bằng BFS trên các phép xoá (mỗi trạng thái là tuple giá trị). Đáp án hợp lệ ⟺ thuộc tập này. Dùng cho test ngẫu nhiên.

In [6]:
def reachable_finals(vals):
    start = tuple(vals)
    seen, finals = {start}, set()
    q = deque([start])
    while q:
        st = q.popleft()
        nxt = []
        for i in range(len(st)):
            s = 0
            for j in range(i, len(st)):
                s += st[j]
                if s == 0:
                    nxt.append(st[:i] + st[j + 1:])
        if not nxt:
            finals.add(st)
        for t in nxt:
            if t not in seen:
                seen.add(t)
                q.append(t)
    return finals


def is_valid(original, result):
    return tuple(result) in reachable_finals(original)

print(sorted(reachable_finals([1, 2, -3, 3, 1])))
assert is_valid([1, 2, -3, 3, 1], [3, 1]) and is_valid([1, 2, -3, 3, 1], [1, 2, 1])

[(1, 2, 1), (3, 1)]


## 4. Ý tưởng nền tảng / trực giác

Tính tay `[1, 2, 3, -3, -2]`. Thêm `P` (gắn với nút dummy và từng nút):

| nút | dummy | 1 | 2 | 3 | -3 | -2 |
|-----|-------|---|---|---|----|----|
| prefix | 0 | 1 | 3 | 6 | 3 | 1 |

- Prefix `3` xuất hiện ở nút `2` và nút `-3` → đoạn **sau** nút `2` tới hết nút `-3` (`[3, -3]`) tổng 0.
- Prefix `1` xuất hiện ở nút `1` và nút `-2` → đoạn sau nút `1` tới hết nút `-2` (`[2, 3, -3, -2]`) tổng 0 — **bao trùm** đoạn trên.

Nếu từ mỗi nút ta "nhảy" tới **lần xuất hiện cuối** của cùng prefix, ta bỏ qua đoạn dài nhất có thể: từ nút `1` (prefix 1) nhảy tới sau nút `-2` → kết quả `[1]`.

In [7]:
vals = [1, 2, 3, -3, -2]
s, pref = 0, [("dummy", 0)]
for v in vals:
    s += v
    pref.append((v, s))
print(pref)

[('dummy', 0), (1, 1), (2, 3), (3, 6), (-3, 3), (-2, 1)]


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** mô phỏng đúng đề — chuyển sang list Python, tìm **một** đoạn liên tiếp tổng 0 bất kỳ (hai vòng lặp), xoá nó, lặp lại đến khi không còn.

In [8]:
def remove_zero_brute(head):
    vals = to_list(head)
    changed = True
    while changed:
        changed = False
        for i in range(len(vals)):
            s = 0
            for j in range(i, len(vals)):
                s += vals[j]
                if s == 0:
                    vals = vals[:i] + vals[j + 1:]
                    changed = True
                    break
            if changed:
                break
    return build(vals)

for vals, want in examples:
    got = to_list(remove_zero_brute(build(vals)))
    print(vals, "->", got, "| hợp lệ:", is_valid(vals, got))
    assert is_valid(vals, got)

[1, 2, -3, 3, 1] -> [3, 1] | hợp lệ: True
[1, 2, 3, -3, 4] -> [1, 2, 4] | hợp lệ: True
[1, 2, 3, -3, -2] -> [1] | hợp lệ: True


**Độ phức tạp:** mỗi lần tìm `O(n²)`, tối đa `n` lần xoá → `O(n³)` = `10^9` với `n = 1000` — quá chậm. Bộ nhớ `O(n)`.

**Nút thắt:** sau mỗi lần xoá ta bắt đầu tìm lại từ đầu, và việc tìm đoạn tổng 0 bằng hai vòng lặp là `O(n²)`.

## 6. Bước đột phá

> **Insight:** đoạn sau nút `A` đến hết nút `B` có tổng 0 ⟺ `prefix(A) == prefix(B)`. Vì vậy, nếu với mỗi tổng tiền tố `p` ta biết nút **cuối cùng** `last[p]` có tổng đó, thì từ bất kỳ nút nào có tổng `p`, ta cho `next` trỏ tới `last[p].next` — bỏ qua đoạn tổng 0 dài nhất bắt đầu sau nó.

Invariant khi duyệt lượt hai (đi theo các `next` đã sửa): nút đang đứng có tổng tiền tố `p` (tính theo danh sách **gốc**) và chính là `last[p]` (lần xuất hiện cuối) — nên tổng tiền tố gốc của các nút được giữ lại **tăng dần theo vị trí** và **đôi một khác nhau**.

Quy tắc "nếu… thì…":

- Lượt 1: duyệt từ dummy, `prefix += val`, `last[prefix] = node` (ghi đè).
- Lượt 2: duyệt lại từ dummy, `prefix += val`, `node.next = last[prefix].next`.

Vì sao tổng tiền tố của nút được giữ không đổi? Ta chỉ bỏ những đoạn tổng 0, nên tổng tiền tố "theo danh sách mới" của nút được giữ bằng tổng tiền tố theo danh sách gốc.

In [9]:
# Minh hoạ last[prefix] trên ví dụ 3
vals = [1, 2, 3, -3, -2]
dummy = ListNode(0, build(vals))
last, s, node = {}, 0, dummy
while node:
    s += node.val
    last[s] = node
    node = node.next
print({p: n.val for p, n in last.items()})   # val của nút cuối có prefix p (dummy có val 0)

{0: 0, 1: -2, 3: -3, 6: 3}


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Với mỗi nút, quét tìm đoạn tổng 0 bắt đầu ngay sau nó

Duyệt `prev` từ dummy. Từ `prev.next`, cộng dồn; mỗi khi tổng bằng 0 tại nút `q`, ghi nhớ `q` (lấy `q` **xa nhất**). Nếu có, `prev.next = q.next` và **không** tiến `prev` (đoạn mới sau `prev` có thể lại tổng 0); ngược lại tiến `prev`.

In [10]:
def remove_zero_scan(head):
    dummy = ListNode(0, head)
    prev = dummy
    while prev:
        s, cur, cut = 0, prev.next, None
        while cur:
            s += cur.val
            if s == 0:
                cut = cur                 # đoạn (prev, cur] tổng 0; tiếp tục tìm xa hơn
            cur = cur.next
        if cut:
            prev.next = cut.next
        else:
            prev = prev.next
    return dummy.next

for vals, _ in examples:
    got = to_list(remove_zero_scan(build(vals)))
    assert is_valid(vals, got), (vals, got)
print("scan OK")

scan OK


Vì sao lấy đoạn **xa nhất** rồi vẫn cần xét lại `prev`? Sau khi bỏ đoạn xa nhất, phần còn lại sau `prev` không có tổng tiền tố (tính từ `prev`) bằng 0 — thực ra đã đảm bảo, nên lần xét lại sẽ không cắt gì và `prev` tiến lên. Giữ vòng lặp như vậy cho an toàn và dễ lập luận.

**Độ phức tạp:** mỗi `prev` quét tới cuối `O(n)` → `O(n²)` = `10^6`, đủ nhanh cho `n ≤ 1000`. Bộ nhớ `O(1)`.

### 7.2 Cách 3 — Hash map hai lượt (lời giải chuẩn)

In [11]:
def remove_zero_two_pass(head):
    dummy = ListNode(0, head)
    last = {}
    s, node = 0, dummy
    while node:                       # lượt 1: nút cuối cùng của mỗi tổng tiền tố
        s += node.val
        last[s] = node
        node = node.next
    s, node = 0, dummy
    while node:                       # lượt 2: nhảy qua đoạn tổng 0 dài nhất
        s += node.val
        node.next = last[s].next
        node = node.next
    return dummy.next

for vals, _ in examples:
    got = to_list(remove_zero_two_pass(build(vals)))
    print(vals, "->", got)
    assert is_valid(vals, got)

[1, 2, -3, 3, 1] -> [3, 1]
[1, 2, 3, -3, 4] -> [1, 2, 4]
[1, 2, 3, -3, -2] -> [1]


**Độ phức tạp:** hai lượt `O(n)`, `O(n)` bộ nhớ cho `dict`.

### 7.3 Cách 4 — Một lượt với dọn dẹp hash map

Duyệt một lần, `seen[prefix] = node`. Khi gặp prefix **đã có** tại nút `A`: đoạn `(A, cur]` tổng 0. Trước khi nối `A.next = cur.next`, phải **xoá khỏi `seen`** các prefix của những nút bị loại (nếu không, sau này có thể nhảy về một nút đã bị xoá).

In [12]:
def remove_zero_one_pass(head):
    dummy = ListNode(0, head)
    seen = {0: dummy}
    s, cur = 0, head
    while cur:
        s += cur.val
        if s in seen:
            A = seen[s]
            # dọn các prefix của nút trong đoạn (A, cur)
            t, node = s, A.next
            while node is not cur:
                t += node.val
                del seen[t]
                node = node.next
            A.next = cur.next
        else:
            seen[s] = cur
        cur = cur.next
    return dummy.next

for vals, _ in examples:
    got = to_list(remove_zero_one_pass(build(vals)))
    print(vals, "->", got)
    assert is_valid(vals, got)

[1, 2, -3, 3, 1] -> [3, 1]
[1, 2, 3, -3, 4] -> [1, 2, 4]
[1, 2, 3, -3, -2] -> [1]


Chi tiết dọn dẹp: bắt đầu `t = s` (prefix của `A`), cộng dần giá trị các nút từ `A.next` đến trước `cur` — mỗi giá trị `t` là prefix của một nút bị xoá, nên `del seen[t]`. Prefix của `cur` bằng `s` (chính là của `A`), giữ nguyên.

**Độ phức tạp:** mỗi nút được thêm vào `seen` và bị xoá khỏi `seen` tối đa một lần → `O(n)` tổng (phân tích khấu hao — amortized). Bộ nhớ `O(n)`.

### 7.4 Cách 5 — Stack trên mảng giá trị (dễ hình dung)

Chuyển sang mảng, giữ `stack` các phần tử được giữ lại, `pos[prefix]` = độ dài stack khi đạt prefix đó. Gặp prefix đã có → cắt stack về độ dài đó và xoá các prefix bị cắt khỏi `pos`. Cuối cùng dựng lại linked list.

In [13]:
def remove_zero_stack(head):
    stack, prefs = [], [0]            # prefs[k] = tổng của stack[:k]
    pos = {0: 0}
    for v in to_list(head):
        s = prefs[-1] + v
        if s in pos:
            k = pos[s]
            for p in prefs[k + 1:]:
                del pos[p]
            del stack[k:]
            del prefs[k + 1:]
        else:
            stack.append(v)
            prefs.append(s)
            pos[s] = len(stack)
    return build(stack)

for vals, _ in examples:
    got = to_list(remove_zero_stack(build(vals)))
    assert is_valid(vals, got)
print("stack OK")

stack OK


**Độ phức tạp:** `O(n)` khấu hao, `O(n)` bộ nhớ. Không sửa con trỏ trên list gốc mà dựng list mới — chấp nhận được với LeetCode.

## 8. TRACE từng bước

Trace cách 3 trên `[1, 2, -3, 3, 1]`.

**Lượt 1** (prefix sau mỗi nút, `last` ghi đè):

| nút | dummy | 1 | 2 | -3 | 3 | 1 |
|-----|-------|---|---|----|---|---|
| prefix | 0 | 1 | 3 | 0 | 3 | 4 |

`last = {0: nút(-3), 1: nút(1), 3: nút(3), 4: nút(1 cuối)}`.

**Lượt 2:** dummy (prefix 0) → `next = last[0].next` = nút `3`. Nút `3` (prefix gốc 3) → `next = last[3].next` = nút `1` cuối. Nút `1` (prefix 4) → `next = None`. Kết quả `[3, 1]`.

In [14]:
def trace_two_pass(vals):
    dummy = ListNode(0, build(vals))
    last, s, node = {}, 0, dummy
    while node:
        s += node.val
        last[s] = node
        node = node.next
    print("last =", {p: ("dummy" if n is dummy else n.val) for p, n in last.items()})
    s, node = 0, dummy
    while node:
        s += node.val
        jump = last[s].next
        name = "dummy" if node is dummy else node.val
        print(f"  tại {name!s:>5} prefix={s:>2} -> next = {jump.val if jump else None}")
        node.next = jump
        node = node.next
    return to_list(dummy.next)

print(trace_two_pass([1, 2, -3, 3, 1]))
print(trace_two_pass([1, 2, 3, -3, -2]))

last = {0: -3, 1: 1, 3: 3, 4: 1}
  tại dummy prefix= 0 -> next = 3
  tại     3 prefix= 3 -> next = 1
  tại     1 prefix= 4 -> next = None
[3, 1]
last = {0: 'dummy', 1: -2, 3: -3, 6: 3}
  tại dummy prefix= 0 -> next = 1
  tại     1 prefix= 1 -> next = None
[1]


Trace cách 4 (một lượt) — in `seen` sau mỗi bước để thấy việc dọn dẹp.

In [15]:
def trace_one_pass(vals):
    dummy = ListNode(0, build(vals))
    seen = {0: dummy}
    s, cur = 0, dummy.next
    while cur:
        s += cur.val
        if s in seen:
            A = seen[s]
            t, node, removed = s, A.next, []
            while node is not cur:
                t += node.val
                del seen[t]
                removed.append(t)
                node = node.next
            A.next = cur.next
            print(f"val={cur.val:>3} prefix={s:>2} ĐÃ CÓ -> cắt đoạn, xoá prefix {removed}; list={to_list(dummy.next)}")
        else:
            seen[s] = cur
            print(f"val={cur.val:>3} prefix={s:>2} mới   -> seen keys={sorted(seen)}")
        cur = cur.next
    return to_list(dummy.next)

print(trace_one_pass([1, 3, 2, -3, -2, 5, 5, -5, 1]))

val=  1 prefix= 1 mới   -> seen keys=[0, 1]
val=  3 prefix= 4 mới   -> seen keys=[0, 1, 4]
val=  2 prefix= 6 mới   -> seen keys=[0, 1, 4, 6]
val= -3 prefix= 3 mới   -> seen keys=[0, 1, 3, 4, 6]
val= -2 prefix= 1 ĐÃ CÓ -> cắt đoạn, xoá prefix [4, 6, 3]; list=[1, 5, 5, -5, 1]
val=  5 prefix= 6 mới   -> seen keys=[0, 1, 6]
val=  5 prefix=11 mới   -> seen keys=[0, 1, 6, 11]
val= -5 prefix= 6 ĐÃ CÓ -> cắt đoạn, xoá prefix [11]; list=[1, 5, 1]
val=  1 prefix= 7 mới   -> seen keys=[0, 1, 6, 7]
[1, 5, 1]


## 9. Code nộp LeetCode

Chọn cách 3 (hash map hai lượt) — ngắn, `O(n)`, không cần dọn dẹp.

In [16]:
from typing import Optional


class Solution:
    def removeZeroSumSublists(self, head: Optional[ListNode]) -> Optional[ListNode]:
        dummy = ListNode(0, head)
        last = {}
        s, node = 0, dummy
        while node:
            s += node.val
            last[s] = node                # nút cuối cùng có tổng tiền tố s
            node = node.next
        s, node = 0, dummy
        while node:
            s += node.val
            node.next = last[s].next      # bỏ qua đoạn tổng 0 dài nhất phía sau
            node = node.next
        return dummy.next


sol = Solution()
for vals, want in examples:
    got = to_list(sol.removeZeroSumSublists(build(vals)))
    print(vals, "->", got)
    assert is_valid(vals, got)
print("Solution OK")

[1, 2, -3, 3, 1] -> [3, 1]
[1, 2, 3, -3, 4] -> [1, 2, 4]
[1, 2, 3, -3, -2] -> [1]
Solution OK


## 10. Vì sao đúng

Gọi `p(x)` là tổng tiền tố **theo danh sách gốc** tại nút `x` (dummy có `p = 0`).

1. **Chỉ xoá đoạn tổng 0.** Khi đứng tại nút `x` với `p(x) = s`, ta nối `x.next = last[s].next`. Các nút bị bỏ là từ `x.next` đến `last[s]`, tổng = `p(last[s]) − p(x) = 0`. Mỗi bước chính là một phép xoá hợp lệ của đề (thực hiện tuần tự từ trái sang phải).
2. **Tổng tiền tố được bảo toàn.** Vì chỉ bỏ các đoạn tổng 0, tổng tiền tố theo danh sách mới của mỗi nút giữ lại bằng `p` gốc của nó → giá trị `s` trong lượt 2 đúng.
3. **Kết quả không còn đoạn tổng 0.** Mỗi nút giữ lại `y` (kể cả dummy) là `last[p(y)]` — lần xuất hiện cuối cùng của tổng đó (vì ta nhảy tới `last[s]` rồi mới đi tiếp; nút tiếp theo được thăm nằm sau `last[s]`). Hai nút giữ lại khác nhau có `p` khác nhau (nếu bằng, nút sau sẽ là `last` và nút trước đã phải nhảy qua nó). Tổng tiền tố đôi một khác nhau ⟹ không có đoạn liên tiếp tổng 0 (mục 3.4).

Vậy kết quả là một trạng thái cuối hợp lệ.

## 11. Time / Space Complexity

Gọi `n` = số nút.

| Cách | Time | Space |
|------|------|-------|
| 1. Mô phỏng | `O(n³)` | `O(n)` |
| 2. Quét từ mỗi nút | `O(n²)` | `O(1)` |
| 3. Hash map hai lượt | `O(n)` | `O(n)` |
| 4. Một lượt + dọn dẹp | `O(n)` khấu hao | `O(n)` |
| 5. Stack trên mảng | `O(n)` khấu hao | `O(n)` |

Hash map: thao tác `O(1)` trung bình. Tổng tiền tố trong `[-10^6, 10^6]` — có thể dùng mảng thay dict nếu muốn.

## 12. Bộ test

### 12.1 Ví dụ + edge case (kiểm tra bằng `is_valid`)

In [17]:
impls = {
    "brute": remove_zero_brute,
    "scan": remove_zero_scan,
    "two_pass": remove_zero_two_pass,
    "one_pass": remove_zero_one_pass,
    "stack": remove_zero_stack,
    "Solution": Solution().removeZeroSumSublists,
}
edge = [
    [0],                    # một nút 0 → rỗng
    [5],
    [1, -1],                # xoá hết
    [0, 0, 0],
    [1, 2, 3],              # không có gì để xoá
    [1, -1, 1, -1],
    [3, -3, 2, -2, 1],
    [2, 0, 2],
    [1, 3, 2, -3, -2, 5, 5, -5, 1],
]
for name, f in impls.items():
    for vals in [v for v, _ in examples] + edge:
        got = to_list(f(build(vals)))
        assert not has_zero_segment(got), (name, vals, got)
        assert is_valid(vals, got), (name, vals, got)
print("all impls valid on", len(examples) + len(edge), "cases")

all impls valid on 12 cases


Cách 3 và cách 2 trả về **cùng một** đáp án cụ thể? Không nhất thiết — đó là lý do ta dùng `is_valid`. Xem các đáp án khác nhau trên một input:

In [18]:
vals = [1, 2, -3, 3, 1]
for name, f in impls.items():
    print(f"{name:9}", to_list(f(build(vals))))
print("mọi trạng thái cuối:", sorted(reachable_finals(vals)))

brute     [3, 1]
scan      [3, 1]
two_pass  [3, 1]
one_pass  [3, 1]
stack     [3, 1]
Solution  [3, 1]
mọi trạng thái cuối: [(1, 2, 1), (3, 1)]


### 12.2 Đối chiếu ngẫu nhiên (seed cố định)

Giá trị trong `[-3, 3]` để thường xuyên có đoạn tổng 0; độ dài ≤ 9 để BFS `reachable_finals` nhanh.

In [19]:
rng = random.Random(1171)
for _ in range(300):
    vals = [rng.randint(-3, 3) for _ in range(rng.randint(1, 9))]
    finals = reachable_finals(vals)
    for name, f in impls.items():
        got = tuple(to_list(f(build(vals))))
        assert got in finals, (name, vals, got)
print("random cross-check OK")

random cross-check OK


### 12.3 Test giới hạn

`n = 1000`: ngẫu nhiên, toàn `0`, dạng `[k, -k, k, -k, ...]` và dạng lồng nhau `[1,2,…,500,-500,…,-1]`.

In [20]:
rng = random.Random(8)
bigs = [
    [rng.randint(-1000, 1000) for _ in range(1000)],
    [0] * 1000,
    [7, -7] * 500,
    list(range(1, 501)) + [-x for x in range(500, 0, -1)],
]
for vals in bigs:
    for name in ["scan", "two_pass", "one_pass", "stack", "Solution"]:
        t0 = time.perf_counter()
        got = to_list(impls[name](build(vals)))
        dt = time.perf_counter() - t0
        assert not has_zero_segment(got)
    print(len(vals), "-> len kết quả", len(got), f"(lần cuối {dt:.3f}s)")

1000 -> len kết quả 151 (lần cuối 0.000s)
1000 -> len kết quả 0 (lần cuối 0.000s)
1000 -> len kết quả 0 (lần cuối 0.000s)
1000 -> len kết quả 0 (lần cuối 0.000s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không dùng dummy: bỏ sót đoạn bắt đầu từ `head`

In [21]:
def wrong_no_dummy(head):
    last, s, node = {}, 0, head
    while node:
        s += node.val
        last[s] = node
        node = node.next
    s, node = 0, head
    while node:
        s += node.val
        node.next = last[s].next
        node = node.next
    return head

vals = [1, -1, 2]
got = to_list(wrong_no_dummy(build(vals)))
print("sai:", got, "| hợp lệ?", is_valid(vals, got), "| đúng:", to_list(Solution().removeZeroSumSublists(build(vals))))
assert not is_valid(vals, got)

sai: [1, -1, 2] | hợp lệ? False | đúng: [2]


**Vì sao sai:** đoạn `[1, -1]` bắt đầu từ `head` cần "nút trước" có tổng tiền tố `0` — chính là dummy. Thiếu dummy, prefix `0` chưa bao giờ được ghi trước `head`. **Sửa:** `dummy = ListNode(0, head)` và duyệt từ dummy.

### Lỗi 2 — Dùng lần xuất hiện **đầu tiên** của prefix trong lượt hai

In [22]:
def wrong_first_occurrence(head):
    dummy = ListNode(0, head)
    first, s, node = {}, 0, dummy
    while node:
        s += node.val
        first.setdefault(s, node)          # giữ lần ĐẦU thay vì lần cuối
        node = node.next
    s, node, steps = 0, dummy, 0
    while node and steps < 20:             # giới hạn bước: code sai này có thể chạy mãi
        s += node.val
        node.next = first[s].next
        node = node.next
        steps += 1
    return dummy.next, steps


def to_list_safe(head, limit=30):
    """Như to_list nhưng dừng sau `limit` nút (phát hiện vòng lặp)."""
    out = []
    while head and len(out) < limit:
        out.append(head.val)
        head = head.next
    return out, head is not None


vals = [1, 2, -3, 3, 1]
head, steps = wrong_first_occurrence(build(vals))
got, has_cycle = to_list_safe(head)
print("số bước lượt 2:", steps, "| danh sách (30 nút đầu):", got, "| có vòng lặp:", has_cycle)
assert has_cycle

số bước lượt 2: 20 | danh sách (30 nút đầu): [1, 2, -3, 1, 2, -3, 1, 2, -3, 1, 2, -3, 1, 2, -3, 1, 2, -3, 1, 2, -3, 1, 2, -3, 1, 2, -3, 1, 2, -3] | có vòng lặp: True


**Vì sao sai:** tại nút `-3` (prefix `0`), `first[0]` là **dummy**, nên `(-3).next = dummy.next = nút 1` — con trỏ quay ngược về phía trước, tạo **vòng lặp** (cycle): lượt 2 chạy mãi, `to_list` không bao giờ dừng. Còn tại các nút xuất hiện lần đầu, `first[s]` là chính nó nên không xoá gì. **Sửa:** ghi đè để lấy lần xuất hiện **cuối** — khi đó mọi bước nhảy đều đi **về phía sau**.

### Lỗi 3 — Một lượt nhưng không dọn `seen`

In [23]:
def wrong_one_pass_no_cleanup(head):
    dummy = ListNode(0, head)
    seen = {0: dummy}
    s, cur = 0, head
    while cur:
        s += cur.val
        if s in seen:
            seen[s].next = cur.next        # quên xoá prefix của các nút bị loại
        else:
            seen[s] = cur
        cur = cur.next
    return dummy.next

vals = [1, 3, 2, -3, -2, 5, 5, -5, 1]
got = to_list(wrong_one_pass_no_cleanup(build(vals)))
print("sai:", got, "| hợp lệ?", is_valid(vals, got), "| đúng (một đáp án):", to_list(remove_zero_one_pass(build(vals))))
assert not is_valid(vals, got)

sai: [1, 5, 5, -5, 1] | hợp lệ? False | đúng (một đáp án): [1, 5, 1]


**Vì sao sai:** sau khi cắt đoạn, các prefix của nút đã bị xoá vẫn nằm trong `seen`; về sau một prefix trùng với chúng sẽ nối vào **nút đã bị loại** → thao tác vô hiệu, đoạn tổng 0 còn sót. **Sửa:** xoá các prefix đó khỏi `seen` (cách 4), hoặc dùng hai lượt (cách 3).

### Lỗi 4 — Chỉ xoá **một** đoạn tổng 0 rồi dừng (không lặp)

In [24]:
def wrong_single_round(head):
    vals = to_list(head)
    for i in range(len(vals)):
        s = 0
        for j in range(i, len(vals)):
            s += vals[j]
            if s == 0:
                return build(vals[:i] + vals[j + 1:])   # xoá xong trả về luôn
    return build(vals)

vals = [1, -1, 2, -2]
got = to_list(wrong_single_round(build(vals)))
print("sai:", got, "| còn đoạn tổng 0?", has_zero_segment(got), "| đúng:", to_list(Solution().removeZeroSumSublists(build(vals))))
assert has_zero_segment(got)

sai: [2, -2] | còn đoạn tổng 0? True | đúng: []


**Vì sao sai:** xoá `[1, -1]` xong vẫn còn `[2, -2]`. Đề yêu cầu **lặp lại** đến khi không còn đoạn nào. **Sửa:** lặp (cách 1), hoặc dùng tổng tiền tố xử lý tất cả trong một/hai lượt.

### Lỗi 5 — Quên đoạn chỉ gồm **một nút 0**

Một nút có `val = 0` tự nó là đoạn tổng 0. Code chỉ tìm đoạn độ dài ≥ 2 (ví dụ vòng `j` bắt đầu từ `i + 1`) sẽ bỏ sót. Với tổng tiền tố, nút `0` có cùng prefix với nút trước nó nên tự động bị xử lý.

In [25]:
def wrong_len_ge_2(head):
    vals = to_list(head)
    changed = True
    while changed:
        changed = False
        for i in range(len(vals)):
            s = vals[i]
            for j in range(i + 1, len(vals)):     # bỏ qua đoạn độ dài 1
                s += vals[j]
                if s == 0:
                    vals = vals[:i] + vals[j + 1:]; changed = True; break
            if changed: break
    return build(vals)

vals = [3, 0, 4]
got = to_list(wrong_len_ge_2(build(vals)))
print("sai:", got, "| đúng:", to_list(Solution().removeZeroSumSublists(build(vals))))
assert has_zero_segment(got)

sai: [3, 0, 4] | đúng: [3, 4]


## 14. Mẹo & nhận dạng pattern

- **"Đoạn con liên tiếp có tổng bằng k"** → tổng tiền tố + hash map: `P[j] − P[i] = k` (LC 560 Subarray Sum Equals K, LC 325, LC 523).
- **Linked list có thể xoá từ đầu** → dummy node.
- **Hai lượt với "lần xuất hiện cuối"** là mẹo hay để xử lý mọi đoạn lồng nhau/chồng nhau mà không cần dọn dẹp.
- **Bài nhiều đáp án** → viết bộ kiểm tra tính chất (không còn đoạn tổng 0) + kiểm tra đạt được (BFS với input nhỏ).
- Khi một lượt duyệt **xoá phần tử khỏi cấu trúc phụ** (dict/stack), nghĩ đến phân tích khấu hao: mỗi phần tử vào/ra một lần.

## 15. Biến thể

1. **Mảng thay vì linked list** — cách 5 (stack) áp dụng trực tiếp.
2. **Xoá đoạn có tổng bằng `k`** (khác 0) — không còn tính "lặp lại an toàn" như tổng 0 (xoá đoạn tổng `k` làm thay đổi tổng tiền tố phía sau), cần định nghĩa lại cẩn thận.
3. **LC 560 Subarray Sum Equals K** — đếm số đoạn tổng `k`.
4. **LC 525 Contiguous Array** — đoạn dài nhất có số 0 và 1 bằng nhau (đổi 0 → −1, tìm prefix trùng xa nhất).
5. **Trả về độ dài lớn nhất có thể** của danh sách cuối — vì nhiều đáp án, câu hỏi tối ưu trở nên thú vị (cần DP).

In [26]:
# Biến thể 4: LC 525 — đoạn dài nhất có số 0 và 1 bằng nhau
def find_max_length(nums):
    first = {0: -1}
    s = best = 0
    for i, x in enumerate(nums):
        s += 1 if x == 1 else -1
        if s in first:
            best = max(best, i - first[s])
        else:
            first[s] = i
    return best

print(find_max_length([0, 1]), find_max_length([0, 1, 0]), find_max_length([0, 0, 1, 0, 0, 0, 1, 1]))
assert find_max_length([0, 0, 1, 0, 0, 0, 1, 1]) == 6

2 2 6


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|------|---------|------|-------|--------------|
| 1. Mô phỏng | Tìm & xoá lặp lại | `O(n³)` | `O(n)` | Kiểm chứng |
| 2. Quét từ mỗi nút | Đoạn tổng 0 xa nhất sau `prev` | `O(n²)` | `O(1)` | Không muốn dùng hash map, `n` nhỏ |
| 3. Hai lượt hash map | `next = last[prefix].next` | `O(n)` | `O(n)` | **Nộp bài** |
| 4. Một lượt + dọn | Cắt khi prefix trùng, xoá prefix cũ | `O(n)` KH | `O(n)` | Dữ liệu dạng luồng |
| 5. Stack mảng | Cắt stack về vị trí prefix | `O(n)` KH | `O(n)` | Mảng, dễ debug |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** `Solution().removeZeroSumSublists(build([2, -2, 3]))` → ?

**Bài 2 (tính tay):** lập bảng prefix và `last` cho `[4, -1, -3, 2, 5, -7]`, rồi suy ra kết quả của cách 3.

**Bài 3 (điền TODO):** hàm `prefix_sums(head)` trả về list tổng tiền tố (bắt đầu bằng 0) của linked list.

**Bài 4 (tự viết lại):** viết lại cách 3 trên **mảng Python** (trả về list), dùng `last[prefix] = chỉ số`.

**Bài 5 (mini challenge):** đếm số đoạn liên tiếp tổng 0 trong mảng ban đầu (LC 560 với `k = 0`) trong `O(n)`.

In [27]:
# Bài 3 — TODO
def prefix_sums(head):
    # TODO
    raise NotImplementedError

# assert prefix_sums(build([1, 2, -3])) == [0, 1, 3, 0]

In [28]:
# Bài 5 — TODO
def count_zero_sum_subarrays(nums):
    # TODO: dùng Counter các tổng tiền tố
    pass

# assert count_zero_sum_subarrays([1, -1, 1, -1]) == 4

## 18. Gợi ý 3 mức

- **Bài 1:** Mức 1: prefix `0, 2, 0, 3`. Mức 2: `last[0]` là nút `-2`. Mức 3: dummy nhảy tới nút `3` → `[3]`.
- **Bài 2:** Mức 1: prefix `0, 4, 3, 0, 2, 7, 0`. Mức 2: `last[0]` = nút cuối (`-7`). Mức 3: dummy nhảy qua tất cả → `[]`.
- **Bài 3:** Mức 1: `out = [0]`. Mức 2: duyệt, `out.append(out[-1] + node.val)`. Mức 3: trả `out`.
- **Bài 4:** Mức 1: `P` độ dài `n + 1`. Mức 2: `last[P[k]] = k`. Mức 3: từ `k = 0`, nhảy `k = last[P[k]]`, giữ `a[k]` (vị trí kế) rồi `k += 1`.
- **Bài 5:** Mức 1: `Counter({0: 1})`. Mức 2: với mỗi prefix `s`, cộng `cnt[s]` vào đáp án rồi `cnt[s] += 1`. Mức 3: độ phức tạp `O(n)`.

## 19. Đáp án tham khảo

In [29]:
from collections import Counter

# Bài 1
assert to_list(Solution().removeZeroSumSublists(build([2, -2, 3]))) == [3]
# Bài 2
assert to_list(Solution().removeZeroSumSublists(build([4, -1, -3, 2, 5, -7]))) == []

# Bài 3
def prefix_sums_sol(head):
    out = [0]
    while head:
        out.append(out[-1] + head.val)
        head = head.next
    return out

assert prefix_sums_sol(build([1, 2, -3])) == [0, 1, 3, 0]

# Bài 4
def remove_zero_array(a):
    P = [0]
    for x in a:
        P.append(P[-1] + x)
    last = {p: k for k, p in enumerate(P)}      # ghi đè → chỉ số cuối
    res, k = [], 0
    while True:
        k = last[P[k]]                          # nhảy tới lần cuối của prefix
        if k == len(a):
            break
        res.append(a[k])                        # phần tử a[k] nằm giữa P[k] và P[k+1]
        k += 1
    return res

rng = random.Random(41)
for _ in range(300):
    a = [rng.randint(-3, 3) for _ in range(rng.randint(1, 8))]
    assert is_valid(a, remove_zero_array(a))
print("bài 1-4 OK")

bài 1-4 OK


In [30]:
# Bài 5
def count_zero_sum_subarrays_sol(nums):
    cnt = Counter({0: 1})
    s = ans = 0
    for x in nums:
        s += x
        ans += cnt[s]
        cnt[s] += 1
    return ans

assert count_zero_sum_subarrays_sol([1, -1, 1, -1]) == 4
rng = random.Random(5)
for _ in range(200):
    a = [rng.randint(-2, 2) for _ in range(rng.randint(0, 10))]
    brute = sum(1 for i in range(len(a)) for j in range(i, len(a)) if sum(a[i:j + 1]) == 0)
    assert count_zero_sum_subarrays_sol(a) == brute
print("bài 5 OK")

bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

```python
dummy = ListNode(0, head)
last, s, node = {}, 0, dummy
while node:                      # lượt 1
    s += node.val
    last[s] = node               # ghi đè → lần cuối
    node = node.next
s, node = 0, dummy
while node:                      # lượt 2
    s += node.val
    node.next = last[s].next
    node = node.next
return dummy.next
```

**Checklist:**

- [ ] Có dummy với `val = 0` (prefix 0 trước `head`)?
- [ ] Lượt 1 **ghi đè** để lấy lần xuất hiện **cuối**?
- [ ] Lượt 2 tính prefix lại từ dummy, cùng thứ tự?
- [ ] Trả về `dummy.next` (head có thể bị xoá)?
- [ ] Nút đơn `0` được xử lý (tự động với prefix)?
- [ ] Test bằng hàm kiểm tra hợp lệ vì có nhiều đáp án?